# ASSIGNMENT GIAI ĐOẠN 1: CHUẨN BỊ DỮ LIỆU VÀ XÂY DỰNG MÔ HÌNH CƠ SỞ
* **Môn học:** Học máy cơ bản (ITA202)
* **Khóa học:** FPT Polytechnic
* **Đề tài:** Dự đoán giá nhà California (California Housing Price Prediction)
* **Vai trò:** Data Scientist

---

### MỤC TIÊU VÀ NHIỆM VỤ GIAI ĐOẠN 1:
1. **Khám phá dữ liệu (EDA):** Kiểm tra cấu trúc dữ liệu thô (Shape, Info, Describe, Missing values) và trực quan hóa phân phối.
2. **Làm sạch dữ liệu:** Xử lý giá trị khuyết thiếu (NaN) ở cột `total_bedrooms` và kỹ nghệ đặc trưng tỷ lệ (Feature Engineering).
3. **Tiền xử lý dữ liệu:** Chuẩn hóa đặc trưng số (StandardScaler), mã hóa biến phân loại `ocean_proximity` (One-Hot Encoding), chia tập Train (80%) và Test (20%).
4. **Huấn luyện mô hình cơ sở:** Xây dựng mô hình Hồi quy tuyến tính (**Linear Regression**).
5. **Đánh giá ban đầu:** Đo lường các chỉ số MAE, MSE, RMSE, R² và trực quan hóa sai số, làm tiền đề chuẩn bị cho **Giai đoạn 2 (Tối ưu hóa và hoàn thiện)**.


In [ ]:
# 1. Nạp các thư viện cần thiết
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập hiển thị
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Sklearn modules cho tiền xử lý và mô hình cơ sở
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✅ Đã nạp thành công các thư viện cho Giai đoạn 1!")


---
## BƯỚC 1: CHỌN VÀ KHÁM PHÁ DỮ LIỆU (EDA)
Tập dữ liệu được chọn là **California Housing**, dữ liệu điều tra dân số thực tế gồm 20,640 khu vực tại California.
Biến mục tiêu cần dự đoán ($y$) là `median_house_value` (Giá nhà trung vị tính bằng USD).


In [ ]:
# Đọc dữ liệu từ file CSV
data_path = os.path.join('data', 'housing.csv')
if not os.path.exists(data_path):
    data_path = 'housing.csv'

df = pd.read_csv(data_path)
print(f"📊 Kích thước dữ liệu (Shape): {df.shape[0]} dòng, {df.shape[1]} cột")
df.head()


In [ ]:
# Kiểm tra kiểu dữ liệu và số lượng non-null
print("📋 Cấu trúc kiểu dữ liệu (Info):")
df.info()


In [ ]:
# Thống kê mô tả (Describe)
print("📈 Thống kê mô tả các đặc trưng số:")
df.describe()


In [ ]:
# Kiểm tra giá trị khuyết thiếu (NaN)
missing_counts = df.isnull().sum()
missing_ratio = (missing_counts / len(df)) * 100
missing_df = pd.DataFrame({'Số lượng khuyết thiếu': missing_counts, 'Tỷ lệ (%)': missing_ratio})
missing_df[missing_df['Số lượng khuyết thiếu'] > 0]


In [ ]:
# Phân phối biến mục tiêu (median_house_value)
plt.figure(figsize=(9, 5))
sns.histplot(df['median_house_value'], bins=50, kde=True, color='royalblue')
plt.title("Phân phối giá nhà trung vị (Median House Value)", fontsize=13, fontweight='bold')
plt.xlabel("Giá trị nhà ($ USD)")
plt.ylabel("Số lượng mẫu (Count)")
plt.tight_layout()
plt.show()


In [ ]:
# Bản đồ vị trí địa lý California theo Kinh độ - Vĩ độ
plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    df['longitude'], df['latitude'],
    c=df['median_house_value'], cmap='jet', alpha=0.4,
    s=df['population'] / 100, label='Dân số (/100)'
)
plt.colorbar(scatter, label='Giá nhà ($ USD)')
plt.title("Bản đồ địa lý giá nhà California", fontsize=13, fontweight='bold')
plt.xlabel("Kinh độ (Longitude)")
plt.ylabel("Vĩ độ (Latitude)")
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
# Ma trận tương quan Pearson
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, linewidths=0.5)
plt.title("Ma trận tương quan Pearson", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Mức độ tương quan với giá nhà:")
print(corr_matrix['median_house_value'].sort_values(ascending=False))


---
## BƯỚC 2: LÀM SẠCH DỮ LIỆU & KỸ NGHỆ ĐẶC TRƯNG

1. **Xử lý giá trị khuyết thiếu (NaN):**
   - Cột `total_bedrooms` bị khuyết **207 mẫu** (~1%).
   - Áp dụng phương pháp điền giá trị trung vị (**Median Imputation** = 435.0) vì phân phối lệch phải, trung vị không bị méo bởi các ngoại lệ lớn.
2. **Kỹ nghệ đặc trưng (Feature Engineering):**
   - `rooms_per_household = total_rooms / households`: Số phòng trung bình trong một hộ gia đình.
   - `bedrooms_per_room = total_bedrooms / total_rooms`: Tỷ lệ phòng ngủ trên tổng số phòng.
   - `population_per_household = population / households`: Số nhân khẩu trung bình trong một hộ gia đình.


In [ ]:
# Tạo đặc trưng mới
df_clean = df.copy()
df_clean['rooms_per_household'] = df_clean['total_rooms'] / df_clean['households']
df_clean['bedrooms_per_room'] = df_clean['total_bedrooms'] / df_clean['total_rooms']
df_clean['population_per_household'] = df_clean['population'] / df_clean['households']

print("Kích thước dữ liệu sau khi thêm đặc trưng:", df_clean.shape)
df_clean[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].describe()


---
## BƯỚC 3: TIỀN XỬ LÝ DỮ LIỆU & CHIA TẬP TRAIN/TEST

1. **Tách dữ liệu:** Tách biến độc lập $X$ và biến mục tiêu $y$ (`median_house_value`).
2. **Chia tập Train/Test:** Tỷ lệ **80% Train** và **20% Test** theo đúng yêu cầu với `random_state=42`.
3. **Chuẩn hóa & Mã hóa:**
   - Thuộc tính số: `SimpleImputer(strategy='median')` -> `StandardScaler()`.
   - Thuộc tính phân loại (`ocean_proximity`): `OneHotEncoder(handle_unknown='ignore')`.


In [ ]:
# Tách X và y
X = df_clean.drop(columns=['median_house_value'])
y = df_clean['median_house_value']

# Chia tập Train (80%) và Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Tập Train: {X_train.shape[0]} mẫu ({X_train.shape[0]/len(X):.0%})")
print(f"Tập Test : {X_test.shape[0]} mẫu ({X_test.shape[0]/len(X):.0%})")

num_features = [col for col in X.columns if col != 'ocean_proximity']
cat_features = ['ocean_proximity']

# Xây dựng ColumnTransformer tiền xử lý
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cat_features)
])

# Fit trên Train và Transform trên cả Train và Test (chống rò rỉ dữ liệu)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print(f"✅ Dữ liệu sau tiền xử lý có {X_train_prepared.shape[1]} đặc trưng đầu vào.")


---
## BƯỚC 4 & 5: HUẤN LUYỆN MÔ HÌNH CƠ SỞ (LINEAR REGRESSION) & ĐÁNH GIÁ BAN ĐẦU

Theo yêu cầu của Giai đoạn 1 đối với bài toán Hồi quy, ta áp dụng mô hình **Hồi quy tuyến tính (Linear Regression)** để làm mô hình cơ sở (Baseline).

Đánh giá mô hình dựa trên các chỉ số chuẩn:
* **MAE (Mean Absolute Error):** Sai số tuyệt đối trung bình (\$ USD)
* **MSE (Mean Squared Error):** Sai số bình phương trung bình
* **RMSE (Root Mean Squared Error):** Căn bậc hai sai số bình phương trung bình (\$ USD)
* **$R^2$ Score:** Hệ số xác định (tỷ lệ phương sai dữ liệu mà mô hình giải thích được)


In [ ]:
# 4. Huấn luyện mô hình Linear Regression
baseline_model = LinearRegression()
baseline_model.fit(X_train_prepared, y_train)

# 5. Dự đoán và đánh giá
y_train_pred = baseline_model.predict(X_train_prepared)
y_test_pred = baseline_model.predict(X_test_prepared)

# Tính toán các chỉ số
train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print("=" * 60)
print("📊 BẢNG KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH CƠ SỞ (LINEAR REGRESSION)")
print("=" * 60)
print(f"Tập Train: RMSE = ${train_rmse:,.2f} | R² = {train_r2:.4f}")
print(f"Tập Test : RMSE = ${test_rmse:,.2f} | R² = {test_r2:.4f}")
print(f"Tập Test : MAE  = ${test_mae:,.2f}")
print(f"Tập Test : MSE  = {test_mse:,.2f}")
print("=" * 60)


In [ ]:
# Trực quan hóa kết quả đánh giá mô hình cơ sở
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị Thực tế vs Dự đoán
axes[0].scatter(y_test, y_test_pred, alpha=0.3, color='royalblue', s=20)
min_v = min(y_test.min(), y_test_pred.min())
max_v = max(y_test.max(), y_test_pred.max())
axes[0].plot([min_v, max_v], [min_v, max_v], 'r--', lw=2, label='Đường lý tưởng (y = x)')
axes[0].set_title("Giá thực tế vs Giá dự đoán (Linear Regression)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Giá thực tế ($ USD)")
axes[0].set_ylabel("Giá dự đoán ($ USD)")
axes[0].legend()

# Đồ thị Phân phối phần dư Residuals
residuals = y_test - y_test_pred
sns.histplot(residuals, bins=50, kde=True, ax=axes[1], color='crimson')
axes[1].axvline(0, color='black', linestyle='--', lw=1.5)
axes[1].set_title("Phân phối sai số phần dư (Residuals)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Phần dư = Thực tế - Dự đoán ($ USD)")
axes[1].set_ylabel("Tần suất")

plt.tight_layout()
plt.show()


---
## KẾT LUẬN GIAI ĐOẠN 1 & ĐỊNH HƯỚNG GIAI ĐOẠN 2

### 1. Đánh giá kết quả Giai đoạn 1:
* Đã hoàn thành trọn vẹn quy trình ML cơ bản từ dữ liệu thô đến mô hình dự đoán đầu tiên.
* Dữ liệu đã được làm sạch triệt để: xử lý 207 giá trị NaN, bổ sung 3 đặc trưng kỹ nghệ hữu ích, chuẩn hóa StandardScaler và One-Hot Encoding.
* Mô hình cơ sở **Linear Regression** đạt $R^2 pprox 0.6353$ và $RMSE pprox \$69,127$ trên tập kiểm thử (Test set).

### 2. Nhận xét về hiện tượng Underfitting:
* Mô hình tuyến tính đơn giản chưa thể nắm bắt hết các mối quan hệ phi tuyến phức tạp của vị trí địa lý và các thuộc tính nhà ở, dẫn đến hiện tượng **Underfitting (Thiếu khớp)**.

### 3. Kế hoạch thực hiện Giai đoạn 2:
* Thử nghiệm các mô hình phi tuyến tính nâng cao: **KNN Regressor**, **Decision Tree**, **Random Forest**.
* Đóng gói toàn bộ quy trình thành **Pipeline** chuyên nghiệp.
* Sử dụng **GridSearchCV** kết hợp Cross-Validation để tối ưu hóa siêu tham số.
* So sánh các mô hình nâng cao với Baseline của Giai đoạn 1 và đóng gói mô hình ra file `.pkl`.
